# custom-sam-peft — GPU Test Runner

Three-tier GPU test runner. Run the tier cell that matches your hardware; the
cleanup trailer runs after every tier (pass or fail) so the next tier starts
from a clean GPU state without a full runtime restart.

**Prereqs:**
1. Runtime → Change runtime type → **T4 GPU** (minimum CC 7.5, e.g. Tesla T4;
   required for the `gpu_t4` and `gpu_bf16` tiers).
2. In Colab Secrets (left sidebar 🔑), add:
   - `HF_TOKEN` — Hugging Face access token with read access to gated `facebook/sam3.1`.
   - `GH_TOKEN` — GitHub fine-grained personal access token with **Contents: Read** on this repo. **Required only if the repo is private**; can be omitted for public repos.
3. **Set `BRANCH` in the next cell** (default is `"main"`).
4. Run setup cells (collapsed below) once per session.
5. Press play on each tier cell in turn. All three tier cells green = PR ready.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SET THE BRANCH HERE before running anything else.
#   - default: "main"
#   - for a PR/feature branch, e.g.: BRANCH = "manual-gpu-pass-44"
# ════════════════════════════════════════════════════════════════════════
BRANCH = "main"

REPO = "NguyenJus/custom-sam-peft"
print(f"REPO   = {REPO}")
print(f"BRANCH = {BRANCH}")

### GPU guard (runs first)

This first code cell asserts a CUDA GPU is available before any install
or test step. Colab can pin the **accelerator class** (GPU vs. CPU) via
notebook metadata (`metadata.colab.accelerator: "GPU"`), but it cannot
pin the **GPU model** — the most common assignment is a T4, but Colab
may serve a V100, A100, L4, or others depending on availability. The
`gpu_t4` tier is calibrated for a T4 (CC ≥ 7.5, ≤ 16 GB VRAM); the guard
warns (does not fail) if a different GPU is detected. CC < 7.5 (Pascal /
GTX 1080 and earlier) is no longer supported — all GPU tests autoskip.


In [ ]:
# GPU guard. Runs FIRST so a misconfigured runtime fails loudly before any
# slow install or test step. Two assertions:
#   1. nvidia-smi must succeed AND report at least one GPU. If it doesn't,
#      the runtime is CPU-only — change Runtime → Change runtime type → GPU.
#   2. If the GPU is not a T4, print a WARN: the gpu_t4 tier's timing and
#      memory assumptions are calibrated for a Colab T4; other GPUs (V100,
#      A100, L4, …) may show different characteristics. The cell does NOT
#      fail in that case — other GPUs are usually fine for gpu_t4 as long
#      as they satisfy CC ≥ 7.5 and ≤ 16 GB VRAM.
import subprocess
import sys

try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        check=True,
        capture_output=True,
        text=True,
    )
except FileNotFoundError as e:
    raise RuntimeError(
        "nvidia-smi not found; this runtime is CPU-only. "
        "Runtime → Change runtime type → GPU (T4 recommended for gpu_t4 tier) → Save."
    ) from e
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        f"nvidia-smi failed (returncode={e.returncode}). stderr: {e.stderr.strip()}"
    ) from e

gpus = [line.strip() for line in out.stdout.splitlines() if line.strip()]
if not gpus:
    raise RuntimeError(
        "nvidia-smi reported no GPUs. "
        "Runtime → Change runtime type → GPU (T4 recommended for gpu_t4 tier) → Save."
    )

print(f"GPU(s) detected: {gpus}")
if not any("T4" in g for g in gpus):
    print(
        f"WARN: detected GPU is not a T4 ({gpus}). The gpu_t4 tier's timing "
        "and memory assumptions are calibrated for a T4; other CC ≥ 7.5 GPUs are "
        "usually fine for gpu_t4 but unverified.",
        file=sys.stderr,
    )

In [ ]:
# Cell 1: Clone + checkout. Done BEFORE any torch/numpy import so the pip
# install in Cell 2 can downgrade numpy without Colab forcing a runtime restart.
import os
import subprocess

try:
    from google.colab import userdata  # type: ignore

    gh_token = userdata.get("GH_TOKEN")
except Exception:
    gh_token = None

clone_url = (
    f"https://{gh_token}@github.com/{REPO}.git" if gh_token else f"https://github.com/{REPO}.git"
)

if not os.path.isdir("custom-sam-peft"):
    subprocess.run(["git", "clone", clone_url], check=True)
subprocess.run(["git", "-C", "custom-sam-peft", "fetch", "--all"], check=True)
subprocess.run(["git", "-C", "custom-sam-peft", "checkout", BRANCH], check=True)
os.chdir("custom-sam-peft")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
# Install runtime + GPU + test deps + pytest-sugar (per-test progress bar).
#
# numpy/scipy/transformers pins must stay on the SAME pip install line so
# pip's resolver honors them in one pass; splitting them re-resolves and
# tries to upgrade numpy past sam3's <2 cap, cascading scipy backtracks.
# torchao>=0.16.0 is pinned because peft 0.19+ raises ImportError on the
# torchao 0.10.0 that Colab preinstalls.
%pip install -e ".[qlora,dev]" \
    "numpy==1.26.4" "scipy==1.13.1" "transformers==5.0.0" \
    "huggingface_hub>=1.15" \
    "torchao>=0.16.0" \
    "pytest-sugar"
!python -c "import custom_sam_peft; print('custom_sam_peft OK:', custom_sam_peft.__file__)"

# Fail-fast check: if Colab preloaded numpy 2.x into the kernel before our
# install ran, the on-disk numpy is now 1.26.4 but sys.modules still holds
# the old 2.x. Auto-restart attempts proved unreliable; instead, raise a
# clear instruction.
import sys

if "numpy" in sys.modules:
    import importlib.metadata

    _in_mem = sys.modules["numpy"].__version__
    _on_disk = importlib.metadata.version("numpy")
    if _in_mem != _on_disk:
        raise RuntimeError(
            f"numpy mismatch: kernel has {_in_mem}, disk has {_on_disk}.\n"
            "Colab pre-imported the old version before our pin took effect.\n"
            "Runtime → Restart session, then Run all (the install re-runs as a no-op)."
        )

In [ ]:
# Cell 4: HF auth (token in Colab Secrets).
import os

from google.colab import userdata

token = userdata.get("HF_TOKEN")
assert token, "HF_TOKEN missing from Colab Secrets. See the prereqs cell."
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token  # huggingface-cli reads this name too

In [ ]:
# Cell 5: Download the SAM 3.1 checkpoint (gated; HF_TOKEN required).
# `huggingface-cli` is deprecated; the new CLI is `hf` (huggingface_hub >= 1.13).
!hf download facebook/sam3.1 --local-dir models/sam3.1

## Test tier cells

Three tiers, each corresponding to a hardware capability band. Run the tier
that matches your environment. The `gpu_t4` tier is the primary release gate
and requires a Colab T4 (CC ≥ 7.5, ≤ 16 GB VRAM); `gpu_bf16` validates
faithful bf16 on CC ≥ 8.0 cards; `gpu_xl` is currently empty and reserved
for future > 16 GB tests.

The long-running `gpu_t4` cell streams the trainer's WARNING-level logs live
(`-s --log-cli-level=WARNING` inside the runner script), so you can see
NaN-class skips or training events as they happen.

### Coverage matrix

**34 GPU-gated tests** (see
[`docs/testing/gpu-test-policy.md`](../docs/testing/gpu-test-policy.md)
for the authoritative per-test breakdown):

| Tier | Marker | Runner arg | Tests | Hardware |
| --- | --- | --- | --- | --- |
| `gpu_t4` | `gpu_t4` | `t4` | 33 | CC ≥ 7.5 AND ≤ 16 GB VRAM (Tesla T4 floor; fp16 band below CC 8.0) |
| `gpu_bf16` | `gpu_bf16` | `bf16` | 1 | CC ≥ 8.0 AND ≤ 16 GB VRAM (faithful non-coerced bf16, e.g. RTX 5070 Ti) |
| `gpu_xl` | `gpu_xl` | `xl` | 0 (reserved) | > 16 GB VRAM; cloud auto-provision (#124) |

**Minimum supported GPU:** Tesla T4 (CC 7.5). Pascal / GTX 1080 / CC < 7.5
is no longer supported — all GPU tests autoskip on those cards.

**gpu_t4 (33):** all structural, forward-pass, predict, and training-loop
tests — load/forward (2), LoRA (4), QLoRA (5), channel adapter (3), predict
nchannel (1), GPU predict base + vram_hint (2), training overfits, QLoRA
resume/quant-state roundtrip, K=8 multiplex forward, K=16 VRAM calibration,
and the end-to-end CLI smoke.

**gpu_bf16 (1):** the bf16-coercion contract test — verifies that CC ≥ 8.0
cards run faithful, non-coerced bf16 numerics.


In [ ]:
%%bash
# colab-min tier — 2 tests. ~5–10 min wall on a Colab T4.
# Minimal Colab proof: install + load SAM 3.1 forward pass + one short QLoRA smoke.
# Runs exactly two node IDs:
#   tests/integration/test_load_sam31_real.py::test_load_sam31_forward_to_canonical
#   tests/gpu/test_real_train_qlora.py::test_qlora_overfits_in_50_steps
# Small surface — validates that the repo installs and the core pipeline
# functions on Colab without running the full gpu_t4 (33 tests) suite.
cleanup() {
    echo ""
    echo "=== GPU CLEANUP ==="
    ORPHANS=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ')
    if [ -n "$ORPHANS" ]; then
        echo "Killing GPU processes still holding memory: $ORPHANS"
        echo "$ORPHANS" | xargs -r kill -9 2>/dev/null
        sleep 1
    fi
    USED=$(nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits | tr -d ' ')
    [ "$USED" -lt 100 ] && echo "GPU memory clean (${USED} MiB used)" || echo "WARN: GPU memory not released (${USED} MiB used)"
    echo ""
    echo "Tier exit code: ${TEST_EXIT:-<interrupted>} (0 = all passed)"
}
trap cleanup EXIT
set -e
git pull --ff-only
set +e
bash scripts/run_gpu_tests.sh colab-min
TEST_EXIT=$?

### bf16 coercion on T4 (finding #139)

On a Colab T4 (Compute Capability 7.5) bf16 is **coerced to fp16** by
`coerce_dtype_for_capability` in `src/custom_sam_peft/runtime/_runtime.py`.
Only CC >= 8.0 cards (e.g. RTX 5070 Ti) run faithful, non-coerced bf16.

The `colab-min` tier above therefore validates load + forward + one QLoRA smoke
under fp16 — correct behaviour on a T4, but not a substitute for the
`gpu_bf16` tier which exercises faithful bf16 numerics on a CC >= 8.0 card.

See [`docs/testing/gpu-test-policy.md`](../docs/testing/gpu-test-policy.md)
for the full tier-to-hardware policy.

In [ ]:
# #193: T4 per-step timing sample (to be recorded in docs/defaults-provenance.md)
#
# The colab-min runner above already ran the QLoRA 50-step smoke.  This cell
# captures a cheap per-step timing measurement for the same smoke test so the
# wall-clock seconds-per-step can be logged for #193.
#
# HOW TO USE:
#   1. Run the colab-min cell above first (or separately).
#   2. Run this cell immediately after to time the same smoke test in isolation.
#   3. Copy the printed "per-step" number into docs/defaults-provenance.md
#      under the "T4 QLoRA smoke timing" section.
#
# NOTE: This cell does NOT execute GPU tests on its own — it re-runs only the
# 50-step QLoRA smoke and wraps it with perf_counter for a clean measurement.
import subprocess
import time

cmd = [
    "python",
    "-m",
    "pytest",
    "tests/gpu/test_real_train_qlora.py::test_qlora_overfits_in_50_steps",
    "-x",
    "--no-header",
    "-q",
    "-o",
    "addopts=",  # strip global --cov-fail-under so subset runs cleanly
]
t0 = time.perf_counter()
result = subprocess.run(cmd, capture_output=False)
elapsed = time.perf_counter() - t0
steps = 50
per_step = elapsed / steps
print(f"\nWall-clock total : {elapsed:.1f}s")
print(f"Steps            : {steps}")
print(f"Per-step (approx): {per_step:.2f}s/step")
print("Record this in docs/defaults-provenance.md — T4 QLoRA smoke timing (#193)")

In [ ]:
%%bash
# gpu_t4 tier — 10 tests. ~30–40 min wall on a Colab T4.
# Requires > 8 GB VRAM and/or bf16-faithful numerics; validated on the Colab T4 (16 GB).
# Covers: training overfits (LoRA 50-step, QLoRA 50-step, QLoRA fast smoke),
#         QLoRA resume / quant-state roundtrip, LoRA + QLoRA predict (run training first),
#         K=8 multiplex forward, K=16 VRAM calibration, and the end-to-end CLI smoke.
# -s --log-cli-level=WARNING streams trainer warnings live (NaN-class skips, VRAM events).
cleanup() {
    echo ""
    echo "=== GPU CLEANUP ==="
    ORPHANS=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ')
    if [ -n "$ORPHANS" ]; then
        echo "Killing GPU processes still holding memory: $ORPHANS"
        echo "$ORPHANS" | xargs -r kill -9 2>/dev/null
        sleep 1
    fi
    USED=$(nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits | tr -d ' ')
    [ "$USED" -lt 100 ] && echo "GPU memory clean (${USED} MiB used)" || echo "WARN: GPU memory not released (${USED} MiB used)"
    echo ""
    echo "Tier exit code: ${TEST_EXIT:-<interrupted>} (0 = all passed)"
}
trap cleanup EXIT
set -e
git pull --ff-only
set +e
bash scripts/run_gpu_tests.sh t4
TEST_EXIT=$?

In [ ]:
%%bash
# gpu_xl tier — currently 0 tests (reserved for > 16 GB / larger-arch runners).
# Requires a cloud-provisioned runner; see #124.
# Run this cell now as a smoke-check that the tier filter collects 0 tests cleanly
# (exit code 5 = "no tests collected" is treated as success by the runner script).
cleanup() {
    echo ""
    echo "=== GPU CLEANUP ==="
    ORPHANS=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ')
    if [ -n "$ORPHANS" ]; then
        echo "Killing GPU processes still holding memory: $ORPHANS"
        echo "$ORPHANS" | xargs -r kill -9 2>/dev/null
        sleep 1
    fi
    USED=$(nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits | tr -d ' ')
    [ "$USED" -lt 100 ] && echo "GPU memory clean (${USED} MiB used)" || echo "WARN: GPU memory not released (${USED} MiB used)"
    echo ""
    echo "Tier exit code: ${TEST_EXIT:-<interrupted>} (0 or 5 = expected)"
}
trap cleanup EXIT
set -e
git pull --ff-only
set +e
bash scripts/run_gpu_tests.sh xl
TEST_EXIT=$?

## Reading test results

Scroll to the bottom of any test cell's output and read pytest's final summary
line (e.g. `========= 4 passed in 87.3s =========`). That line is the
pass/fail signal. The cleanup trailer below it confirms GPU state for the
next cell.

If a cell fails:
1. The cleanup trailer still ran — GPU memory is reclaimed; you don't need
   to restart the runtime.
2. Read the traceback above the summary line.
3. Push a fix locally, re-press the same cell. The `git pull --ff-only` at
   the top picks up the new commit.
